# F3-matrices — Session 2: Multiplication and Gram Matrices

*One class session, roughly 85 minutes. Prerequisite: Session 1 of this
unit (matrix action, linearity, columns-as-basis-images), F2-vectors
(dot products, norms, cosine similarity), and F1 broadcasting.*

**This session:** multiplying two matrices by hand (row·column, the shape
rule, and why the definition is what it is), the `@` operator in NumPy,
multiplication as *composition of machines*, the exam's **banned-`@`
register** — computing products with nothing but elementwise multiplies,
broadcasting, and axis sums — and the Gram matrix, the all-pairs
dot-product table that turns a stack of row vectors into a similarity
map.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Row Times Column, By Hand

**Motivation.**
Session 1's machines will need to be chained — apply one map, then
another.
The bookkeeping for "chain two machines" is a product of their matrices,
and the exam expects you to compute small products with bare hands, fast
and without slips.

**Definition.**
For $A$ of shape $(n, k)$ and $B$ of shape $(k, m)$, the **product**
$C = AB$ has shape $(n, m)$ and entries
$$C_{ij} \;=\; (\text{row } i \text{ of } A) \cdot (\text{column } j \text{ of } B)
      \;=\; \sum_{t=0}^{k-1} A_{it}\, B_{tj} .$$
One dot product per output entry — $n \cdot m$ dot products, each of
length $k$.

**The shape rule.**
$$(n, k) \cdot (k, m) \;\to\; (n, m):$$
inner dimensions must *match* (they are the length being dotted over) and
then they *cancel*, leaving the outer pair.
If the inner dimensions differ, the product is undefined — there is
nothing to dot.

**Worked example (square).**
$A = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}$,
$B = \begin{pmatrix} 0 & 1 \\ 5 & -1 \end{pmatrix}$.
$$AB = \begin{pmatrix}
1 \cdot 0 + 2 \cdot 5 & 1 \cdot 1 + 2 \cdot (-1) \\
3 \cdot 0 + 4 \cdot 5 & 3 \cdot 1 + 4 \cdot (-1)
\end{pmatrix} = \begin{pmatrix} 10 & -1 \\ 20 & -1 \end{pmatrix}.$$
Now the other order:
$$BA = \begin{pmatrix}
0 \cdot 1 + 1 \cdot 3 & 0 \cdot 2 + 1 \cdot 4 \\
5 \cdot 1 - 1 \cdot 3 & 5 \cdot 2 - 1 \cdot 4
\end{pmatrix} = \begin{pmatrix} 3 & 4 \\ 2 & 6 \end{pmatrix}.$$
$AB \ne BA$ — **order matters**, and Section 3 explains geometrically
why it must.

**Worked example (rectangular).**
$P = \begin{pmatrix} 1 & 0 & 2 \\ 0 & 1 & -1 \end{pmatrix}$ ($(2,3)$) and
$Q = \begin{pmatrix} 1 & 1 \\ 2 & 0 \\ 0 & 3 \end{pmatrix}$ ($(3,2)$).
Shapes: $(2,3)(3,2) \to (2,2)$.
$$PQ = \begin{pmatrix}
1 + 0 + 0 & 1 + 0 + 6 \\
0 + 2 + 0 & 0 + 0 - 3
\end{pmatrix} = \begin{pmatrix} 1 & 7 \\ 2 & -3 \end{pmatrix}.$$
The reverse order is *also* defined here — $(3,2)(2,3) \to (3,3)$ — and
does not even have the same shape.
And $(2,3) \cdot (2,3)$?
Inner dimensions 3 and 2: undefined.

In [ ]:
A = np.array([[1., 2.], [3., 4.]])
B = np.array([[0., 1.], [5., -1.]])

# Hand-check AB entry by entry with explicit dot products.
for i in range(2):
    for j in range(2):
        print(f"C[{i},{j}] = row {i} . col {j} =",
              A[i, 0] * B[0, j] + A[i, 1] * B[1, j])

### Checkpoint 1

1. By hand: compute $\begin{pmatrix} 1 & 2 \\ 3 & 0 \end{pmatrix}
   \begin{pmatrix} 2 & 1 \\ 1 & 1 \end{pmatrix}$.
2. Shapes: which of $(3,2)(2,2)$, $(2,2)(3,2)$, $(1,4)(4,1)$,
   $(4,1)(1,4)$ are defined, and what shape does each defined product
   have?
3. In the product $C = AB$ with $A$ of shape $(n, k)$: how many scalar
   multiplications does computing all of $C$ take, if $B$ has shape
   $(k, m)$?
   (Count $k$ multiplications per dot product.)

## 2. `@` in NumPy

**The operator.**
NumPy spells the matrix product `A @ B` (equivalently `np.matmul(A, B)`).
It applies exactly Section 1's definition and *enforces* the shape rule —
mismatched inner dimensions raise an error instead of a wrong answer.

**Matrix @ vector.**
`M @ x` with `M` of shape $(n, k)$ and `x` of shape $(k,)$ returns the
shape-$(n,)$ action $Mx$ — the same computation Session 1 wrote as
`(M * x).sum(axis=1)`.
Two spellings of one definition; you now own both.

**The trap next door.**
`A * B` is *elementwise* multiplication (F1), a completely different
animal: it needs same-shaped (or broadcastable) operands and never sums
anything.
Confusing `*` with `@` produces well-shaped nonsense for square
matrices — no error message, just wrong numbers.
Section 7 dissects this.

In [ ]:
print("AB via @:")
print(A @ B)                      # matches the hand computation
print("BA via @:")
print(B @ A)                      # different -- order matters

P = np.array([[1., 0., 2.], [0., 1., -1.]])
Q = np.array([[1., 1.], [2., 0.], [0., 3.]])
print("PQ (2,2):")
print(P @ Q)
print("QP (3,3) -- not even the same shape:")
print(Q @ P)

x = np.array([2., 1.])
print("M @ x            :", np.array([[1., 2.], [3., -1.]]) @ x)
print("(M * x).sum(ax=1):", (np.array([[1., 2.], [3., -1.]]) * x).sum(axis=1))

In [ ]:
print("A * B (elementwise -- NOT the matrix product):")
print(A * B)
print("A @ B (matrix product):")
print(A @ B)

try:
    P @ np.array([[1., 2.], [3., 4.]])     # (2,3) @ (2,2): inner 3 vs 2
except ValueError as err:
    print("shape rule enforced:", err)

### Checkpoint 2

1. Predict `A @ v` by hand for
   $A = \begin{pmatrix} 1 & -1 \\ 2 & 0 \end{pmatrix}$, $v = (3, 1)$,
   then the shape of `A @ A @ v`.
2. Without running: for the $2 \times 2$ matrices $A, B$ above, which
   entries of `A * B` and `A @ B` happen to agree?
   Why is the agreement a coincidence?
3. `X` has shape `(5, 3)` and `Y` has shape `(3, 3)`.
   What are the shapes of `X @ Y` and `Y @ X` (careful)?

## 3. Multiplication Is Composition

**Motivation.**
Why *this* definition, with its dot products and its fussy shape rule?
Because it is exactly the bookkeeping for chaining machines.

**The claim.**
For any input $x$:
$$(AB)\,x \;=\; A\,(B x).$$
Run $B$ first, feed its output to $A$ — or apply the single machine
$AB$ — identical results.
Reason via the column secret: column $j$ of $AB$ should be where the
composite sends $e_j$; and indeed $A(B e_j) = A \cdot (\text{col } j$ of
$B)$, which is precisely row-dot-column — the definition.

**Reading order.**
In $ABx$, the machine *nearest the vector acts first*: $B$, then $A$.
Products read right to left, like function composition $f(g(x))$.

**Order matters, geometrically.**
Shear-then-stretch and stretch-then-shear genuinely differ.
With $S = \begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}$ and
$D = \begin{pmatrix} 2 & 0 \\ 0 & 1 \end{pmatrix}$:
$$SD = \begin{pmatrix} 2 & 1 \\ 0 & 1 \end{pmatrix} \qquad
DS = \begin{pmatrix} 2 & 2 \\ 0 & 1 \end{pmatrix};$$
on $x = (0, 1)$: $SDx = (1, 1)$ but $DSx = (2, 1)$.
The plot makes the disagreement visible on a whole shape.

**Association is free.**
$(AB)C = A(BC)$ — both sides are the single machine "run $C$, then $B$,
then $A$", so you may drop the parentheses.
Which *grouping you compute first* is still your choice, and Section 7
shows the cost can differ by orders of magnitude.

In [ ]:
S = np.array([[1., 1.], [0., 1.]])   # shear
Dm = np.array([[2., 0.], [0., 1.]])  # stretch

x = np.array([0., 1.])
print("S(D x) :", S @ (Dm @ x), "   (SD) x :", (S @ Dm) @ x)
print("D(S x) :", Dm @ (S @ x), "   (DS) x :", (Dm @ S) @ x)

house = np.array([[0., 0.], [2., 0.], [2., 1.5], [1., 2.5],
                  [0., 1.5], [0., 0.]])

def apply_points(M, pts):
    return (M[None, :, :] * pts[:, None, :]).sum(axis=2)

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2))
panels = [("original", house),
          ("stretch, then shear: S D", apply_points(S @ Dm, house)),
          ("shear, then stretch: D S", apply_points(Dm @ S, house))]
for ax, (name, mapped) in zip(axes, panels):
    ax.plot(mapped[:, 0], mapped[:, 1], marker="o", markersize=3)
    ax.set_title(name, fontsize=10)
    ax.set_aspect("equal")
    ax.grid(True, linewidth=0.3)
    ax.set_xlim(-1, 6)
    ax.set_ylim(-0.5, 3)
plt.tight_layout()
plt.show()

### Checkpoint 3

1. A pipeline applies machine $B$ first, then machine $A$.
   Which single matrix represents the whole pipeline, and where does the
   input vector sit in the expression?
2. By hand, with $S$ and $D$ above: compute $SD$ and $DS$, then apply
   each to $(1, 1)$.
3. True or false, one line each:
   (a) $(AB)C = A(BC)$ whenever the shapes allow;
   (b) $AB = BA$ whenever the shapes allow.

## 4. The Banned-`@` Register

**The exam's move.**
Round 1's constrained-coding tasks sometimes *ban the easy road*: a
zero-points clause naming some or all of
`@`, `np.matmul`, `np.dot`, `.T`, and loops.
The permitted route is always the same trio — **elementwise multiply,
broadcasting, axis sums** — and the skill being graded is whether you
can rebuild a product from its definition.
You already own every ingredient from F1; what follows is the assembly
manual.

**The full product, banned-style.**
For $A\,(n, k)$ and $B\,(k, m)$, entry $C_{ij} = \sum_t A_{it} B_{tj}$
needs a 3-D scaffold with one axis per index:

| expression | shape | meaning |
|---|---|---|
| `A[:, :, None]` | $(n, k, 1)$ | $A_{it}$, ready to broadcast over $j$ |
| `B[None, :, :]` | $(1, k, m)$ | $B_{tj}$, ready to broadcast over $i$ |
| product | $(n, k, m)$ | cell $[i, t, j] = A_{it} B_{tj}$ — every term of every dot product |
| `.sum(axis=1)` | $(n, m)$ | collapse the shared $t$ axis: $C_{ij}$ |

$$\texttt{C = (A[:, :, None] * B[None, :, :]).sum(axis=1)}$$

The middle axis is the *contraction* axis — the one the shape rule says
must match, and the one the sum eats.
Everything in this register is a variation on that sentence.

**Row-wise variants you will reuse constantly:**

- **matrix · vector** — `y = (M * x).sum(axis=1)` (Session 1's opener);
- **many points as rows** — for `pts` of shape $(p, k)$ under `M` of
  shape $(n, k)$: `out = (M[None, :, :] * pts[:, None, :]).sum(axis=2)`,
  result $(p, n)$;
- **all pairwise row dot products** of `W` $(n, d)$:
  `(W[:, None, :] * W[None, :, :]).sum(axis=2)`, result $(n, n)$ —
  Section 5 gives this one a name.

**Working habit.**
Write the target sum in indices first ($C_{ij} = \sum_t \dots$), give
every index its own axis, then let the sum collapse the shared one.
Predict each intermediate shape *before* running — in this register,
shape prediction is the whole game.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
A2 = rng.normal(0, 1, (4, 3))
B2 = rng.normal(0, 1, (3, 5))

C_banned = (A2[:, :, None] * B2[None, :, :]).sum(axis=1)
C_easy = A2 @ B2
print("intermediate cube shape:", (A2[:, :, None] * B2[None, :, :]).shape)  # (4, 3, 5)
print("result shape           :", C_banned.shape)                           # (4, 5)
print("max |banned - @|       :", np.abs(C_banned - C_easy).max())          # ~0

# matrix . vector, banned-style
M3 = rng.normal(0, 1, (2, 3))
v3 = rng.normal(0, 1, 3)
print("Mv banned:", (M3 * v3).sum(axis=1), "  Mv @:", M3 @ v3)

### Checkpoint 4

1. Write the banned-route expression for `A (3, 2) @ B (2, 4)` and state
   the shape of the intermediate 3-D array.
2. `pts` has shape `(7, 2)` and `M` shape `(2, 2)`.
   What is the shape of `pts[:, None, :] * M[None, :, :]`, which axis do
   you sum, and what comes out?
3. Why does summing over `axis=1` in
   `(A[:, :, None] * B[None, :, :])` — rather than axis 0 or 2 —
   produce the matrix product?
   Answer in terms of the indices $i, t, j$.

## 5. The Gram Matrix

**Motivation.**
Stack $n$ vectors as the rows of $W$ ($(n, d)$): sensor channels, survey
responses, direction lists.
The most useful single object you can compute from the stack is the
table of *all pairwise dot products* — every row dotted with every row.

**Definition.**
The **Gram matrix** of $W$ is the $(n, n)$ matrix
$$G = W W^{\mathsf T}, \qquad G_{ij} = w_i \cdot w_j,$$
where $w_i$ is row $i$ and $W^{\mathsf T}$ (the transpose, `W.T` in
NumPy) is $W$ flipped so its rows become columns — exactly what the
shape rule needs: $(n, d)(d, n) \to (n, n)$.

**Two properties, free of charge (both are F2 facts about dots):**

- **Symmetry:** $G_{ij} = w_i \cdot w_j = w_j \cdot w_i = G_{ji}$.
- **Diagonal = squared lengths:**
  $G_{ii} = w_i \cdot w_i = \lVert w_i \rVert^2$.

Any claimed Gram computation that fails either property is wrong before
you check a single off-diagonal entry — cheap tests, use them.

**Worked example (by hand).**
$W = \begin{pmatrix} 1 & 0 \\ 1 & 1 \end{pmatrix}$:
$G = \begin{pmatrix} 1 & 1 \\ 1 & 2 \end{pmatrix}$
(diagonal: lengths$^2$ are 1 and 2; off-diagonal: $w_0 \cdot w_1 = 1$).

---

**Worked exam-style example 1 (constrained coding, banned-`@`
register).**
Here is a full problem in the register the practice set (p07) and the
real paper use, solved step by step.

> Write `gram(W)` computing the Gram matrix $G = W W^{\mathsf T}$ of
> `W (n, d)` — that is, $G_{ij}$ = dot product of rows $i$ and $j$.
> **Banned (zero points): `@`, `np.matmul`, `np.dot`, `.T`, `np.einsum`,
> loops.**
> For the seeded `W (5, 3)` below, compute `G`, and `diag_gap` = max abs
> difference between `np.diagonal(G)` and the squared row lengths of
> `W`.

*Solution reasoning.*
The target is $G_{ij} = \sum_t W_{it} W_{jt}$ — indices $i, j$ survive,
$t$ is contracted.
Give each its own axis: `W[:, None, :]` is $(n, 1, d)$ carrying
$W_{it}$; `W[None, :, :]` is $(1, n, d)$ carrying $W_{jt}$; the product
is $(n, n, d)$ with cell $[i, j, t] = W_{it}W_{jt}$; summing the last
axis (the shared $t$) leaves $(n, n)$.
Note the contrast with the full-product idiom of Section 4: there the
contracted axis sat in the *middle*; here both factors carry their row
index first and the shared axis is *last* — always sum the axis the
indices share, wherever it lives.
For `diag_gap`: squared row lengths are `(W ** 2).sum(axis=1)` — no
transpose, no dot, no loop.

In [ ]:
def gram(W):
    # G_ij = sum_t W_it W_jt  --  banned register: broadcast, multiply, sum.
    return (W[:, None, :] * W[None, :, :]).sum(axis=2)


rng = np.random.default_rng(SEED)
W = rng.uniform(-2, 2, (5, 3))

G = gram(W)
diag_gap = np.abs(np.diagonal(G) - (W ** 2).sum(axis=1)).max()
sym_gap = np.abs(G - gram(W)[np.arange(5)[None, :], np.arange(5)[:, None]]).max()

print("G shape :", G.shape)
print("diag_gap:", diag_gap)             # ~0: diagonal = squared row lengths
print("sym_gap :", sym_gap)              # ~0: G equals its own flip (index grids)
print("check vs the (allowed, off-exam) easy route:",
      np.abs(G - W @ W.T).max())

*Why the symmetry check above needed no `.T`:* indexing with the
broadcasted grids `np.arange(5)[None, :]` and `np.arange(5)[:, None]`
reads $G_{ji}$ in place of $G_{ij}$ — an index-grid flip, the standard
workaround when transpose-style calls are on the banned list (p07 asks
for exactly this move).

### Checkpoint 5

1. By hand: the Gram matrix of
   $W = \begin{pmatrix} 2 & 0 \\ 1 & 2 \\ 0 & -1 \end{pmatrix}$
   (shape $(3, 2)$ — how big is $G$?).
2. A claimed Gram matrix has $G_{01} = 3$ and $G_{10} = -3$.
   Verdict, instantly?
3. In the banned-register `gram`, state the shape of
   `W[:, None, :] * W[None, :, :]` for `W (6, 4)` and say what its cell
   `[2, 5, 1]` contains.

## 6. Cosines: Normalizing the Gram Matrix

**Motivation.**
Raw dot products conflate two questions: *do these rows point the same
way?* and *how long are they?*
A long row posts big dot products against everyone, aligned or not.
F2's fix was the cosine; here it is applied to the whole table at once.

**Definition.**
The **cosine matrix** of $W$ divides each Gram entry by the two row
lengths:
$$C_{ij} = \frac{G_{ij}}{\lVert w_i \rVert\, \lVert w_j \rVert}
         = \cos\theta_{ij},$$
the cosine of the angle between rows $i$ and $j$ (rows assumed nonzero).
All the F2 readings apply entrywise: $C_{ij} = 1$ — same direction;
$-1$ — opposite; $0$ — orthogonal; the diagonal is all 1s.

**Computing it, register-safe.**
Everything needed is already inside $G$:
row lengths are `np.sqrt(np.diagonal(G))`, and the denominator table
$\lVert w_i \rVert\, \lVert w_j \rVert$ is a broadcast product
`norms[:, None] * norms[None, :]` — no transposes, no loops.

**Worked example.**
Below, row 2 of a seeded stack is built as $3 \times$ row 0 plus a whiff
of noise.
In $G$, entry $G_{02}$ is far from the largest number in its row —
length hides the kinship.
In $C$, entry $C_{02} \approx 1$ stands out immediately: direction
revealed, length discarded.

In [ ]:
rng = np.random.default_rng(SEED)
W6 = rng.normal(0, 1, (4, 6))
W6[2] = 3.0 * W6[0] + rng.normal(0, 0.01, 6)   # a long near-copy of row 0

G6 = gram(W6)
norms = np.sqrt(np.diagonal(G6))
C6 = G6 / (norms[:, None] * norms[None, :])

np.set_printoptions(precision=3, suppress=True)
print("Gram matrix G (raw dots -- row 2's length dominates):")
print(G6)
print("cosine matrix C (direction only):")
print(C6)
print("C[0, 2] =", round(C6[0, 2], 4), " <- the kinship, unmissable")

### Checkpoint 6

1. $G_{ij} = 12$, $\lVert w_i \rVert = 4$, $\lVert w_j \rVert = 3$.
   Compute $C_{ij}$ and interpret it.
2. What are the diagonal entries of any cosine matrix, and why?
3. Rows $w_i = (3, 0)$ and $w_j = (0, 2)$: compute $G_{ij}$ and
   $C_{ij}$.
   What geometric relation (F2 vocabulary) did the cosine just certify?

## 7. Common Pitfalls II

**Pitfall 1 — `*` where `@` belongs (and shape-error illiteracy).**
For square same-size matrices, `A * B` runs happily and returns
well-shaped nonsense — the silent version of the bug.
For rectangular operands it usually crashes, and the error message names
the *broadcast* failure, not the matrix product you thought you were
computing.
Learn to read both symptoms:

In [ ]:
A = np.array([[1., 2.], [3., 4.]])
B = np.array([[0., 1.], [5., -1.]])

print("A * B (silent nonsense for a composition):")
print(A * B)                      # BROKEN if a product was intended
print("A @ B (the product):")
print(A @ B)

P23 = np.ones((2, 3))
Q32 = np.ones((3, 2))
try:
    P23 * Q32                     # BROKEN: elementwise needs matching shapes
except ValueError as err:
    print("P23 * Q32 ->", err)
print("P23 @ Q32 shape:", (P23 @ Q32).shape, " <- the product is fine: (2,3)(3,2)->(2,2)")

The tell: a broadcast error mentioning "operands could not be broadcast"
means you wrote elementwise `*`; a matmul error names the mismatched
inner dimensions.
Fix: decide *first* whether you mean a composition/product (`@`) or an
entrywise scaling (`*`); the two agree only by coincidence.

**Pitfall 2 — grouping blindness: $(AB)C$ vs $A(BC)$ cost.**
Association guarantees the same *answer*; it says nothing about the same
*work*.
Each product $(n,k)(k,m)$ costs about $n \cdot k \cdot m$
multiplications.
For a chain $A\,(500, 2)$, $B\,(2, 500)$, $C\,(500, 3)$:

- $(AB)C$: first $(500, 2)(2, 500) \to 500 \cdot 2 \cdot 500 = 500{,}000$,
  then $(500, 500)(500, 3) \to 750{,}000$ — about **1.25 million**
  multiplications, plus a fat $(500, 500)$ intermediate;
- $A(BC)$: first $(2, 500)(500, 3) \to 3{,}000$, then
  $(500, 2)(2, 3) \to 3{,}000$ — about **6 thousand**, biggest
  intermediate a puny $(2, 3)$.

Same matrix, ~200× less work: pick the grouping that keeps the
intermediates *small*.

In [ ]:
import time

rng = np.random.default_rng(SEED)
A_ = rng.normal(0, 1, (500, 2))
B_ = rng.normal(0, 1, (2, 500))
C_ = rng.normal(0, 1, (500, 3))

t0 = time.perf_counter()
for _ in range(200):
    left = (A_ @ B_) @ C_          # BROKEN grouping: (500,500) intermediate
t_left = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(200):
    right = A_ @ (B_ @ C_)         # fixed grouping: (2,3) intermediate
t_right = time.perf_counter() - t0

print("max |difference|:", np.abs(left - right).max())    # same answer
print(f"(AB)C: {t_left:.4f}s   A(BC): {t_right:.4f}s   "
      f"speedup ~{t_left / t_right:.0f}x")

**Pitfall 3 — reading raw Gram entries as similarity.**
"Row 3 has the biggest dot product with row 0, so row 3 is the most
similar."
Broken: a long, half-aligned row beats a short, perfectly aligned one on
raw dots.
Rank by *cosine* when the question is direction:

In [ ]:
W_demo = np.array([[1., 0.],     # the query row
                   [0.9, 0.1],   # short, almost the same direction
                   [10., 8.]])   # long, visibly off-direction

G_demo = gram(W_demo)
n_demo = np.sqrt(np.diagonal(G_demo))
C_demo = G_demo / (n_demo[:, None] * n_demo[None, :])

print("raw dots with row 0   :", G_demo[0])   # BROKEN ranking: row 2 'wins' (10 > 0.9)
print("cosines with row 0    :", C_demo[0])   # fixed ranking: row 1 wins (0.994 > 0.78)

Raw $G$ answers "how much aligned mass"; cosine $C$ answers "how
aligned".
Choose by the question, and say which you chose.

### Checkpoint 7

1. A teammate computes a composition of two $3 \times 3$ machines as
   `A * B` and gets no error.
   Why not, what is wrong anyway, and what single test vector argument
   would expose the bug?
   (Hint: compare `(A * B) @ x` with `A @ (B @ x)`.)
2. For the chain $u\,(1, 800)$, $M\,(800, 800)$, $v\,(800, 1)$: count
   the approximate multiplications for $(uM)v$ and $u(Mv)$, and name
   every intermediate's shape.
   Which grouping would you compute?

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. $\begin{pmatrix} 1 \cdot 2 + 2 \cdot 1 & 1 \cdot 1 + 2 \cdot 1 \\
   3 \cdot 2 + 0 & 3 \cdot 1 + 0 \end{pmatrix}
   = \begin{pmatrix} 4 & 3 \\ 6 & 3 \end{pmatrix}$.
2. $(3,2)(2,2) \to (3,2)$ ✓; $(2,2)(3,2)$ undefined (inner 2 vs 3);
   $(1,4)(4,1) \to (1,1)$ ✓; $(4,1)(1,4) \to (4,4)$ ✓.
3. $n \cdot m$ dot products of length $k$: about $n \cdot k \cdot m$
   multiplications — the number Section 7's grouping game is played
   with.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $Av = (1 \cdot 3 - 1 \cdot 1,\; 2 \cdot 3 + 0) = (2, 6)$;
   `A @ A @ v` is still shape `(2,)` (each `@` maps length 2 to
   length 2).
2. No entry is computed the same way — `(A*B)[i,j]` is the single
   product $A_{ij}B_{ij}$, while `(A@B)[i,j]` is a two-term dot; any
   numeric agreement would be pure accident of the values.
   (Here e.g. `(A*B)[0,0] = 0` but `(A@B)[0,0] = 10`.)
3. `X @ Y` is `(5, 3)`; `Y @ X` is undefined — inner dimensions
   $3$ and $5$ do not match.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. The single matrix is $AB$ (first-applied machine written nearest the
   vector): the pipeline is $x \mapsto A(Bx) = (AB)x$.
2. $SD = \begin{pmatrix} 2 & 1 \\ 0 & 1 \end{pmatrix}$,
   $DS = \begin{pmatrix} 2 & 2 \\ 0 & 1 \end{pmatrix}$;
   on $(1,1)$: $SD(1,1) = (3, 1)$, $DS(1,1) = (4, 1)$.
3. (a) True — both sides are the same three-stage machine
   (associativity).
   (b) False — Section 1's $AB \ne BA$ is a counterexample; order of
   machines matters.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. `(A[:, :, None] * B[None, :, :]).sum(axis=1)`; the intermediate is
   `(3, 2, 4)`.
2. Shape `(7, 2, 2)`; sum `axis=2` (the shared input-coordinate axis);
   out comes `(7, 2)` — each row a mapped point.
3. Cell $[i, t, j]$ holds $A_{it}B_{tj}$; the product definition sums
   over the shared index $t$, which lives on axis 1.
   Summing axis 0 or 2 would add over $i$ or $j$ — quantities the
   definition keeps separate.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. $G$ is $(3, 3)$:
   $G = \begin{pmatrix} 4 & 2 & 0 \\ 2 & 5 & -2 \\ 0 & -2 & 1 \end{pmatrix}$
   (diagonal 4, 5, 1 = squared lengths; $w_0 \cdot w_1 = 2$,
   $w_0 \cdot w_2 = 0$, $w_1 \cdot w_2 = -2$).
2. Wrong instantly: a Gram matrix is symmetric, $G_{01}$ must equal
   $G_{10}$.
3. Shape `(6, 6, 4)`; cell `[2, 5, 1]` is $W_{2,1} \cdot W_{5,1}$ — one
   term of the dot product between rows 2 and 5.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $C_{ij} = 12 / (4 \cdot 3) = 1$: the rows point in exactly the same
   direction (one is a positive multiple of the other).
2. All 1s: $C_{ii} = G_{ii}/\lVert w_i\rVert^2 = 1$ — every row is
   perfectly aligned with itself.
3. $G_{ij} = 3 \cdot 0 + 0 \cdot 2 = 0$, so $C_{ij} = 0$: the rows are
   orthogonal — and the cosine says so regardless of the lengths 3
   and 2.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. No error because `*` happily multiplies two same-shaped arrays
   entrywise; but the result is not the composition's matrix.
   Any generic test vector $x$ exposes it:
   `(A * B) @ x` differs from `A @ (B @ x)` for almost every input
   (e.g. any $x$ with a nonzero where the entrywise and dot recipes
   disagree).
2. $(uM)v$: $(1,800)(800,800) \to 640{,}000$ mults with intermediate
   $(1, 800)$; then $(1,800)(800,1) \to 800$ — total ≈ 640,800.
   $u(Mv)$: $(800,800)(800,1) \to 640{,}000$ with intermediate
   $(800, 1)$; then $(1,800)(800,1) \to 800$ — total ≈ 640,800.
   Same work here (the fat factor $M$ must be touched once either
   way) — the grouping game pays off when a *small* inner product can
   collapse the chain early, as in the $(500,2)(2,500)(500,3)$ example;
   either grouping is fine in this one.

</details>